# Selective Copying — Mamba implementation gate

**Project: `transformer-vs-ssm` (sequence-modeling blog series). Unrelated to any competition.**

## What this notebook decides

This repo runs a bespoke pure-PyTorch Mamba (`src/models/mamba.py`), not `mamba-ssm`.
Before any MQAR result from it can be interpreted, the implementation has to demonstrate
the canonical capability that selectivity was introduced to provide.

Selective Copying is that test: data tokens sit at **randomised** positions, so no fixed
set of taps can solve it. The control is the same model with position-wise selection
ablated (`--ablate`, which freezes delta/B/C to their sequence means).

| Run | Meaning |
|---|---|
| Mamba, selective | the real model |
| Mamba, ablated | selection disabled — the control |
| Transformer | positive control on the training setup itself |

**Verdict logic**
- selective >> ablated  -> selectivity works; proceed to MQAR sweeps
- selective ~= ablated  -> selection is not functioning; **stop and fix the model**
- transformer also fails -> the *task/training* config is wrong, not the model

## Kaggle settings
- Accelerator: **GPU T4 x1**
- Internet: **ON** (for `git clone`)
- Save Version -> Run All

Runtime is short (small models, T=64). Nothing here touches enwik8 or any checkpoint.

In [ ]:
# Cell 1: Clone (or refresh) the repo
import os, subprocess, sys

REPO = "https://github.com/nvaidyan1/transformer-vs-ssm.git"
if not os.path.exists("transformer-vs-ssm"):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir("/kaggle/working/transformer-vs-ssm")
subprocess.run(["git", "pull", "--ff-only"], check=False)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Cell 2: Dependencies
!pip install -q pyyaml pytest 2>&1 | tail -1
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Cell 3: Task + model tests must pass before anything is trained.
# A green suite here is what makes a negative gate result attributable to the model.
!python -m pytest tests/ -q 2>&1 | tail -5

In [ ]:
# Cell 4: Sanity — look at one example of the task being learned
!python scripts/inspect_selective_copy.py --seq_len 64 --n_data 4 --vocab_size 64 --seed 0 --n_examples 2

In [ ]:
# Cell 5: Shared settings
TASK_ARGS = "n_data=4,seq_len=64"
STEPS     = 4000
BATCH     = 32
SEEDS     = [0, 1, 2]          # multiple seeds: a single run is not evidence
OUT       = "/kaggle/working/gate"

import os
os.makedirs(OUT, exist_ok=True)
print(f"task_args={TASK_ARGS}  steps={STEPS}  batch={BATCH}  seeds={SEEDS}")

In [ ]:
# Cell 6: Mamba — SELECTIVE (the real model)
for s in SEEDS:
    !python src/train_task.py --config configs/recall/mamba_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {STEPS} \
        --batch_size {BATCH} --seed {s} --device cuda \
        --out {OUT}/mamba_selective_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 7: Mamba — ABLATED (control: no position-wise selectivity)
for s in SEEDS:
    !python src/train_task.py --config configs/recall/mamba_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {STEPS} \
        --batch_size {BATCH} --seed {s} --device cuda --ablate \
        --out {OUT}/mamba_ablated_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 8: Transformer — positive control on the training setup
for s in SEEDS:
    !python src/train_task.py --config configs/recall/transformer_small.yaml \
        --task selective_copy --task_args {TASK_ARGS} --steps {STEPS} \
        --batch_size {BATCH} --seed {s} --device cuda \
        --out {OUT}/transformer_seed{s}.json 2>&1 | tail -4

In [ ]:
# Cell 9: Verdict
import glob, json
import numpy as np

rows = {}
for f in sorted(glob.glob(f"{OUT}/*.json")):
    r = json.load(open(f))
    key = f"{r['arch']}{' (ablated)' if r['ablate'] else ''}"
    rows.setdefault(key, []).append(r["final_val_acc"])

chance = json.load(open(glob.glob(f"{OUT}/*.json")[0]))["chance_acc"]
print(f"{'run':24s} {'mean acc':>9s} {'min':>7s} {'max':>7s}   (chance {chance:.3f})")
print("-" * 58)
for k, v in sorted(rows.items()):
    print(f"{k:24s} {np.mean(v):>9.4f} {np.min(v):>7.4f} {np.max(v):>7.4f}")

sel = np.mean(rows.get("mamba", [0]))
abl = np.mean(rows.get("mamba (ablated)", [0]))
tra = np.mean(rows.get("transformer", [0]))

print()
if tra < 0.5:
    print("INCONCLUSIVE — the transformer also failed. The task/training config is at "
          "fault, not the Mamba implementation. Raise steps or simplify the task.")
elif sel > 0.9 and sel - abl > 0.2:
    print(f"PASS — selective {sel:.3f} vs ablated {abl:.3f}. Selectivity is functioning; "
          "MQAR results from this implementation are interpretable.")
elif sel - abl < 0.1:
    print(f"FAIL — selective {sel:.3f} ~= ablated {abl:.3f}. Selection is not doing work. "
          "Fix src/models/mamba.py BEFORE running any MQAR sweep.")
else:
    print(f"PARTIAL — selective {sel:.3f}, ablated {abl:.3f}, transformer {tra:.3f}. "
          "Inspect the curves before proceeding.")

In [ ]:
# Cell 10: Save results as notebook output
import shutil
shutil.make_archive("/kaggle/working/selective_copy_gate_results", "zip", OUT)
print("saved -> /kaggle/working/selective_copy_gate_results.zip")
!ls -la /kaggle/working/*.zip